# 04 — Análise dos Indicadores Nutricionais

Nesta etapa são realizadas as análises finais do projeto utilizando as tabelas da camada Gold.
O objetivo é responder às perguntas definidas no início do trabalho sobre a evolução da obesidade e do excesso de peso no Brasil, considerando diferenças entre homens e mulheres ao longo do tempo.
As análises são realizadas a partir do modelo dimensional construído anteriormente.

## 1. Preparação da visão analítica

Para facilitar as consultas, a tabela fato é relacionada às dimensões de país, tempo, sexo e indicador.
O resultado é uma visão analítica contendo as informações descritivas necessárias para as análises.

In [0]:
from pyspark.sql import functions as F

fact_nutrition = spark.table(
    "workspace.default.gold_fact_nutrition"
)

dim_country = spark.table(
    "workspace.default.gold_dim_country"
)

dim_time = spark.table(
    "workspace.default.gold_dim_time"
)

dim_sex = spark.table(
    "workspace.default.gold_dim_sex"
)

dim_indicator = spark.table(
    "workspace.default.gold_dim_indicator"
)

In [0]:
analytics = (
    fact_nutrition
    .join(dim_country, "country_id")
    .join(dim_time, "time_id")
    .join(dim_sex, "sex_id")
    .join(dim_indicator, "indicator_id")
    .select(
        "country",
        "country_code",
        "year",
        "sex",
        "indicator",
        "prevalence_pct"
    )
)

display(analytics.limit(20))

## 2. Evolução da obesidade no Brasil

A primeira análise busca verificar como a prevalência estimada de obesidade evoluiu no Brasil ao longo do período disponível e se o comportamento foi diferente entre homens e mulheres.

In [0]:
obesity_brazil = (
    analytics
    .filter(
        (F.col("country_code") == "BRA") &
        (F.col("indicator") == "Obesity")
    )
    .select(
        "year",
        "sex",
        "prevalence_pct"
    )
    .orderBy(
        "year",
        "sex"
    )
)

display(obesity_brazil)

Databricks visualization. Run in Databricks to view.

## 3. Evolução do excesso de peso no Brasil

A segunda análise avalia a evolução da prevalência estimada de excesso de peso, novamente comparando homens e mulheres ao longo do período.

In [0]:
overweight_brazil = (
    analytics
    .filter(
        (F.col("country_code") == "BRA") &
        (F.col("indicator") == "Overweight")
    )
    .select(
        "year",
        "sex",
        "prevalence_pct"
    )
    .orderBy(
        "year",
        "sex"
    )
)

display(overweight_brazil)

Databricks visualization. Run in Databricks to view.

## 4. Diferença entre homens e mulheres

Além de observar as séries separadamente, foi calculada a diferença entre a prevalência feminina e masculina em cada ano.
Valores positivos indicam prevalência maior entre mulheres, enquanto valores negativos indicam prevalência maior entre homens.

In [0]:
obesity_difference = (
    obesity_brazil
    .groupBy("year")
    .pivot("sex")
    .agg(F.first("prevalence_pct"))
    .withColumn(
        "difference_female_male",
        F.col("Female") - F.col("Male")
    )
    .orderBy("year")
)

display(obesity_difference)

Databricks visualization. Run in Databricks to view.

### Comparação entre o início e o final da série

Para facilitar a interpretação da mudança ao longo do período, são comparados os primeiros e últimos anos disponíveis na base.

In [0]:
display(
    obesity_brazil
    .filter(
        F.col("year").isin(1980, 2024)
    )
    .orderBy(
        "year",
        "sex"
    )
)

In [0]:
obesity_evolution = (
    obesity_brazil
    .groupBy("sex")
    .agg(
        F.first(
            F.when(F.col("year") == 1980, F.col("prevalence_pct")),
            ignorenulls=True
        ).alias("prevalence_1980"),

        F.first(
            F.when(F.col("year") == 2024, F.col("prevalence_pct")),
            ignorenulls=True
        ).alias("prevalence_2024")
    )
    .withColumn(
        "absolute_change",
        F.col("prevalence_2024") -
        F.col("prevalence_1980")
    )
)

display(obesity_evolution)

### Mudança no excesso de peso entre 1980 e 2024

A mesma comparação é realizada para o indicador de excesso de peso.

In [0]:
overweight_evolution = (
    overweight_brazil
    .groupBy("sex")
    .agg(
        F.first(
            F.when(F.col("year") == 1980, F.col("prevalence_pct")),
            ignorenulls=True
        ).alias("prevalence_1980"),

        F.first(
            F.when(F.col("year") == 2024, F.col("prevalence_pct")),
            ignorenulls=True
        ).alias("prevalence_2024")
    )
    .withColumn(
        "absolute_change",
        F.col("prevalence_2024") -
        F.col("prevalence_1980")
    )
)

display(overweight_evolution)

## 5. Comparação com países da América do Sul

Para contextualizar os resultados brasileiros, os indicadores de 2024 são comparados aos de outros países da América do Sul.
A comparação é realizada utilizando os mesmos indicadores e metodologia presentes no conjunto de dados.

In [0]:
south_america_codes = [
    "ARG",
    "BOL",
    "BRA",
    "CHL",
    "COL",
    "ECU",
    "GUY",
    "PRY",
    "PER",
    "SUR",
    "URY",
    "VEN"
]

In [0]:
obesity_sa_female_2024 = (
    analytics
    .filter(
        (F.col("country_code").isin(south_america_codes)) &
        (F.col("year") == 2024) &
        (F.col("indicator") == "Obesity") &
        (F.col("sex") == "Female")
    )
    .select(
        "country",
        "prevalence_pct"
    )
    .orderBy(
        F.col("prevalence_pct").desc()
    )
)

display(obesity_sa_female_2024)

Databricks visualization. Run in Databricks to view.

In [0]:
comparison_rank = (
    analytics
    .filter(
        (F.col("country_code").isin(south_america_codes)) &
        (F.col("year") == 2024) &
        (F.col("indicator") == "Obesity")
    )
    .groupBy(
        "country",
        "sex"
    )
    .agg(
        F.first("prevalence_pct").alias("prevalence_pct")
    )
    .orderBy(
        "sex",
        F.col("prevalence_pct").desc()
    )
)

display(comparison_rank)

## 6. Resumo dos indicadores brasileiros

A tabela abaixo apresenta os indicadores disponíveis para o Brasil em 2024, permitindo uma visão consolidada da situação mais recente presente na base.

In [0]:
brazil_2024 = (
    analytics
    .filter(
        (F.col("country_code") == "BRA") &
        (F.col("year") == 2024)
    )
    .select(
        "year",
        "sex",
        "indicator",
        "prevalence_pct"
    )
    .orderBy(
        "indicator",
        "sex"
    )
)

display(brazil_2024)

## Conclusão das análises

As análises realizadas mostram uma mudança expressiva nos indicadores nutricionais brasileiros ao longo do período de 1980 a 2024.
A prevalência estimada de obesidade aumentou para ambos os sexos. Entre as mulheres, passou de 8,95% para 33,95%, enquanto entre os homens aumentou de 3,50% para 26,05%. A diferença entre os sexos, que era de aproximadamente 5,44 pontos percentuais em 1980, chegou a aproximadamente 7,90 pontos percentuais em 2024.
O excesso de peso apresentou comportamento semelhante de crescimento, porém com maior aumento entre os homens. A prevalência masculina passou de 23,52% para 63,42%, enquanto a feminina passou de 34,75% para 66,48%. Como consequência, a diferença entre os sexos nesse indicador caiu de aproximadamente 11,24 para 3,06 pontos percentuais.
Na comparação da obesidade feminina com países selecionados da América do Sul em 2024, o Brasil apresentou prevalência estimada de 33,95%, ocupando a oitava posição entre os 12 países analisados.
De forma geral, os resultados indicam crescimento consistente dos dois indicadores analisados durante o período disponível, embora existam diferenças na evolução observada entre homens e mulheres.